In [ ]:
import pandas as pd
from common.db import Database
from common.const import CONST

In [ ]:
db = Database(CONST.DB_PATH)

In [7]:
year = 2027
file = str(year) + "_IHSAA.csv"

In [ ]:
# load schools and enrollments
# Match by the stable myIHSAA "SchoolId" UUID first, falling back to name
# for schools that don't have a myihsaa_id on file yet. IHSAA has renamed
# the same school across export years before (e.g. "Trinity School at
# Greenlawn" -> "Trinity Academy at Greenlawn" -> "Trinity Academy South
# Bend"), and matching by name alone silently created a duplicate school
# row each time that happened.
# Any newly added schools get their logo (next cell) and long/lat (last cell)
# filled in automatically -- no manual lookup needed.

df = pd.read_csv(file)

for index, row in df.iterrows():
    name = row["SchoolName"]
    myihsaa_id = row["SchoolId"]

    school_id = db.get_school_id_by_myihsaa_id(myihsaa_id)
    if school_id is None:
        school_id = db.get_school_id(name)

    if school_id == None:
        type = row["School Type"]
        nickname = row["Nickname"]
        address = row["Address1"];
        city = row["City"]
        zip = row["ZipCode"]
        db.insert_school(name, "", type, nickname, address, city, zip, myihsaa_id=myihsaa_id)
        school_id = db.get_school_id_by_myihsaa_id(myihsaa_id)
    else:
        # Keep the UUID current even for schools matched by name (backfills
        # legacy rows, and re-confirms it for rows already tagged).
        db.update_school_myihsaa_id(school_id, myihsaa_id)

    db.insert_school_enrollment(school_id, year, row["Enrollment"])

In [ ]:
# Scrape school logos from myIHSAA, convert each to this app's single
# canonical web format (common/logo.py -- WebP, capped at
# CONST.SCHOOL_LOGO_MAX_DIMENSION, whatever the source format was), and save
# at frontend/static/<CONST.SCHOOL_LOGO_STATIC_SUBDIR>/<school_id>.<CONST.SCHOOL_LOGO_EXT>,
# named by our own school_id rather than a sanitized school name, so there's
# a single, stable naming scheme independent of however myIHSAA spells the
# name this year. There is no DB column tracking this -- the file's
# existence on disk at that exact path IS the "has a logo" signal.
#
# Matches by the stable myIHSAA "SchoolId" UUID first, falling back to
# school_name for rows that don't have a myihsaa_id on file yet -- a school
# with neither match yet (not in our DB at all) is skipped and reported.
# Safe to re-run: already-converted logos are skipped.
import os
import requests

from common.logo import convert_logo_to_webp

API_BASE = "https://myihsaa-prod-ams.azurewebsites.net/api/school-directory"
SEARCH_URL = f"{API_BASE}/search"
LOGO_URL = lambda school_id: f"{API_BASE}/{school_id}/logo"
SEARCH_BODY = {"limit": -1, "page": 0, "count": 1000, "ihsaaDistrict": None}
HEADERS = {
    "Accept": "application/json, text/plain, */*",
    "Content-Type": "application/json",
    "Referer": "https://www.myihsaa.net/",
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
}

LOGO_DIR = os.path.join(CONST.WEB_DIR, "frontend", "static", CONST.SCHOOL_LOGO_STATIC_SUBDIR)
os.makedirs(LOGO_DIR, exist_ok=True)


def find_school_id(myihsaa_id, name):
    school_id = db.get_school_id_by_myihsaa_id(myihsaa_id)
    if school_id is not None:
        return school_id
    return db.get_school_id(name)


def has_existing_logo(school_id):
    return os.path.exists(os.path.join(LOGO_DIR, f"{school_id}.{CONST.SCHOOL_LOGO_EXT}"))


print("Fetching school directory from myIHSAA...")
resp = requests.post(SEARCH_URL, json=SEARCH_BODY, headers=HEADERS, timeout=30)
resp.raise_for_status()
schools = resp.json().get("items", [])
print(f"  API returned {len(schools)} schools\n")

converted = skipped = no_logo = errors = 0
unmatched = []

for i, school in enumerate(schools, 1):
    name = school["name"]
    myihsaa_id = school["id"]
    has_logo = school.get("hasLogo", False)

    school_id = find_school_id(myihsaa_id, name)
    if school_id is None:
        unmatched.append(name)
        continue

    if has_logo:
        if has_existing_logo(school_id):
            skipped += 1
        else:
            try:
                logo_resp = requests.get(LOGO_URL(myihsaa_id), headers=HEADERS, timeout=30)
                logo_resp.raise_for_status()
                is_svg = "svg" in logo_resp.headers.get("content-type", "")
                webp_bytes = convert_logo_to_webp(logo_resp.content, is_svg=is_svg)

                full_path = os.path.join(LOGO_DIR, f"{school_id}.{CONST.SCHOOL_LOGO_EXT}")
                with open(full_path, "wb") as f:
                    f.write(webp_bytes)
                converted += 1
                print(f"  [{i:3d}/{len(schools)}] converted {school_id}.{CONST.SCHOOL_LOGO_EXT} ({name})")
            except Exception as exc:
                print(f"  [{i:3d}/{len(schools)}] ERROR {name}: {exc}")
                errors += 1
    else:
        no_logo += 1

    db.conn.execute(
        "UPDATE school SET myihsaa_id = ? WHERE school_id = ?",
        (myihsaa_id, school_id),
    )

db.conn.commit()

print(f"\nDone. converted={converted}, skipped={skipped} (already on disk), no_logo={no_logo}, errors={errors}, unmatched={len(unmatched)}")
if unmatched:
    print("  Unmatched myIHSAA names (no matching school.myihsaa_id or school_name -- check for a naming mismatch):")
    for name in unmatched:
        print(f"    - {name}")

In [ ]:
# Geocode any school missing longitude/latitude, using the free US Census
# Bureau geocoder (no API key required). Safe to re-run: only schools with
# a NULL latitude/longitude are looked up.
import time

CENSUS_GEOCODE_URL = "https://geocoding.geo.census.gov/geocoder/locations/onelineaddress"


def geocode_address(address, city, state="IN", zip_code=""):
    one_line = f"{address}, {city}, {state} {zip_code}".strip()
    params = {
        "address": one_line,
        "benchmark": "Public_AR_Current",
        "format": "json",
    }
    resp = requests.get(CENSUS_GEOCODE_URL, params=params, timeout=20)
    resp.raise_for_status()
    matches = resp.json().get("result", {}).get("addressMatches", [])
    if not matches:
        return None, None
    coords = matches[0]["coordinates"]
    return coords["y"], coords["x"]  # latitude, longitude


missing_df = pd.read_sql_query(
    "SELECT school_id, school_name, address, city, zip FROM school WHERE latitude IS NULL OR longitude IS NULL",
    db.conn,
)
print(f"Schools missing coordinates: {len(missing_df)}")

geocoded = 0
failed = []
for _, row in missing_df.iterrows():
    lat, lon = geocode_address(row["address"], row["city"], zip_code=str(row["zip"] or ""))
    if lat is not None:
        db.conn.execute(
            "UPDATE school SET latitude = ?, longitude = ? WHERE school_id = ?",
            (lat, lon, row["school_id"]),
        )
        geocoded += 1
        print(f"  {row['school_name']}: ({lat}, {lon})")
    else:
        failed.append(row["school_name"])
    time.sleep(0.5)  # be polite to the free API

db.conn.commit()

print(f"\nGeocoded {geocoded} school(s).")
if failed:
    print("Could not geocode (check the address manually):")
    for name in failed:
        print(f"  - {name}")